### 스키마 정의 ###

In [2]:
import os
import re
import ipaddress

from datetime import datetime
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

In [3]:
def find_error_log():
    """
    프로젝트 구조와 현재 실행 위치를 기준으로 error log를 찾는다.

    환경변수 ERROR_LOG_PATH가 설정돼 있으면 해당 경로를 우선 사용한다.
    """

    env_path = os.getenv("ERROR_LOG_PATH")

    candidates = []

    if env_path:
        candidates.append(
            Path(env_path).expanduser()
        )

    candidates.extend([
        Path("../data/raw/error.log"),
        Path("../data/raw/error_log.txt"),
        Path("data/raw/error.log"),
        Path("data/raw/error_log.txt"),
        Path("error.log"),
        Path("error_log.txt"),
    ])

    for candidate in candidates:
        resolved_path = candidate.resolve()

        if resolved_path.is_file():
            return resolved_path

    checked_paths = "\n".join(
        f"- {candidate.resolve()}"
        for candidate in candidates
    )

    raise FileNotFoundError(
        "error log 파일을 찾지 못했습니다.\n"
        f"확인한 경로:\n{checked_paths}"
    )


error_file_path = find_error_log()


# data/raw/error.log 구조라면 data 폴더를 기준으로 사용
if error_file_path.parent.name == "raw":
    data_root = error_file_path.parent.parent
else:
    data_root = error_file_path.parent


interim_dir = Path(
    os.getenv(
        "ERROR_INTERIM_DIR",
        data_root / "interim",
    )
).resolve()

output_dir = Path(
    os.getenv(
        "ERROR_OUTPUT_DIR",
        data_root / "output",
    )
).resolve()


interim_dir.mkdir(
    parents=True,
    exist_ok=True,
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


parsed_csv_path = (
    interim_dir
    / "error_parsed.csv"
)

failure_csv_path = (
    output_dir
    / "error_parse_failures.csv"
)

timestamp_missing_csv_path = (
    output_dir
    / "error_timestamp_missing.csv"
)


print(f"입력 파일: {error_file_path}")
print(f"전체 파싱 결과: {parsed_csv_path}")
print(f"실패 결과: {failure_csv_path}")
print(f"Timestamp 조사 결과: {timestamp_missing_csv_path}")

입력 파일: C:\Users\seoyeon\log-dlq-pipeline\data\raw\error.log
전체 파싱 결과: C:\Users\seoyeon\log-dlq-pipeline\data\interim\error_parsed.csv
실패 결과: C:\Users\seoyeon\log-dlq-pipeline\data\output\error_parse_failures.csv
Timestamp 조사 결과: C:\Users\seoyeon\log-dlq-pipeline\data\output\error_timestamp_missing.csv


In [4]:
with open("../data/raw/error.log", encoding='utf-8', errors='replace') as f:
    lines = [l.rstrip("\n") for l in f if l.strip()]

print(f"총 라인 수: {len(lines)}")

총 라인 수: 629


#### 1. 전체 라인 preview ####

In [6]:
for line in lines[:20]:
    print(line)

[Thu Jun 12 13:35:25.818494 2025] [suexec:notice] [pid 1947900:tid 140420159084864] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec)
AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive globally to suppress this message
[Thu Jun 12 13:35:25.840609 2025] [lbmethod_heartbeat:notice] [pid 1947900:tid 140420159084864] AH02282: No slotmem from mod_heartmonitor
[Thu Jun 12 13:35:25.841319 2025] [http2:warn] [pid 1947900:tid 140420159084864] AH02951: mod_ssl does not seem to be enabled
[Thu Jun 12 13:35:25.843764 2025] [mpm_event:notice] [pid 1947900:tid 140420159084864] AH00489: Apache/2.4.37 (Rocky Linux) configured -- resuming normal operations
[Thu Jun 12 13:35:25.843784 2025] [core:notice] [pid 1947900:tid 140420159084864] AH00094: Command line: '/usr/sbin/httpd -D FOREGROUND'
[Thu Jun 12 13:35:37.052218 2025] [autoindex:error] [pid 1947909:tid 140419562436352] [client 210.110.68.35

#### 2. 라인 별 브래킷 [...] 개수 count -> 구조 유형 분류 ####

In [8]:
from collections import Counter

bracket_counts = Counter(line.count("[") for line in lines)
print("브래킷 개수 별 라인 수: ", bracket_counts)

for n in bracket_counts:
    print(f"\n---브래킷 {n}개인 라인 샘플 ---")
    samples = [l for l in lines if l.count("[") -- n][:3]
    for s in samples:
        print(s)

브래킷 개수 별 라인 수:  Counter({4: 459, 3: 152, 0: 18})

---브래킷 3개인 라인 샘플 ---
[Thu Jun 12 13:35:25.818494 2025] [suexec:notice] [pid 1947900:tid 140420159084864] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec)
AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive globally to suppress this message
[Thu Jun 12 13:35:25.840609 2025] [lbmethod_heartbeat:notice] [pid 1947900:tid 140420159084864] AH02282: No slotmem from mod_heartmonitor

---브래킷 0개인 라인 샘플 ---
[Thu Jun 12 13:35:25.818494 2025] [suexec:notice] [pid 1947900:tid 140420159084864] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec)
[Thu Jun 12 13:35:25.840609 2025] [lbmethod_heartbeat:notice] [pid 1947900:tid 140420159084864] AH02282: No slotmem from mod_heartmonitor
[Thu Jun 12 13:35:25.841319 2025] [http2:warn] [pid 1947900:tid 140420159084864] AH02951: mod_ssl does not seem to be enabled

---브래킷 4개인 라인 샘플 ---
[Thu Jun 1

#### 3. 라인 [,] 기준으로 쪼갠 후 각 조각 확인 ####

In [10]:
for line in lines[:10]:
    parts = re.findall(r'\[([^\]]*)\]', line)   # 브래킷 안 내용들만 추출
    remainder = re.sub(r'\[[^\]]*\]', '', line).strip()  # 브래킷 뺀 나머지
    print("brackets: ", parts)
    print("remainder: ", remainder)
    print("---")

brackets:  ['Thu Jun 12 13:35:25.818494 2025', 'suexec:notice', 'pid 1947900:tid 140420159084864']
remainder:  AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec)
---
brackets:  []
remainder:  AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive globally to suppress this message
---
brackets:  ['Thu Jun 12 13:35:25.840609 2025', 'lbmethod_heartbeat:notice', 'pid 1947900:tid 140420159084864']
remainder:  AH02282: No slotmem from mod_heartmonitor
---
brackets:  ['Thu Jun 12 13:35:25.841319 2025', 'http2:warn', 'pid 1947900:tid 140420159084864']
remainder:  AH02951: mod_ssl does not seem to be enabled
---
brackets:  ['Thu Jun 12 13:35:25.843764 2025', 'mpm_event:notice', 'pid 1947900:tid 140420159084864']
remainder:  AH00489: Apache/2.4.37 (Rocky Linux) configured -- resuming normal operations
---
brackets:  ['Thu Jun 12 13:35:25.843784 2025', 'core:notice', 'pid 1947900:tid 14042015908

#### 4. 브래킷 안 내용 중 콜론 유무 별로 구분 ####

In [12]:
first_bracket_contents = [re.findall(r'\[([^\]]*)\]', l) for l in lines]
for parts in first_bracket_contents[:15]:
    print(parts)

['Thu Jun 12 13:35:25.818494 2025', 'suexec:notice', 'pid 1947900:tid 140420159084864']
[]
['Thu Jun 12 13:35:25.840609 2025', 'lbmethod_heartbeat:notice', 'pid 1947900:tid 140420159084864']
['Thu Jun 12 13:35:25.841319 2025', 'http2:warn', 'pid 1947900:tid 140420159084864']
['Thu Jun 12 13:35:25.843764 2025', 'mpm_event:notice', 'pid 1947900:tid 140420159084864']
['Thu Jun 12 13:35:25.843784 2025', 'core:notice', 'pid 1947900:tid 140420159084864']
['Thu Jun 12 13:35:37.052218 2025', 'autoindex:error', 'pid 1947909:tid 140419562436352', 'client 210.110.68.35:59629']
['Thu Jun 12 16:43:50.991473 2025', 'autoindex:error', 'pid 1947909:tid 140419308443392', 'client 210.110.68.35:56126']
['Thu Jun 12 16:56:59.210399 2025', 'autoindex:error', 'pid 1947909:tid 140418427635456', 'client 210.110.64.41:32198']
['Thu Jun 12 16:58:25.662484 2025', 'autoindex:error', 'pid 1948138:tid 140419182618368', 'client 220.67.241.146:55672']
['Thu Jun 12 16:59:32.899367 2025', 'autoindex:error', 'pid 194813

#### 5. 메시지 앞 부분 'AH#####' 있는 라인/없는 라인 비율

In [14]:
has_code = sum(1 for l in lines if re.search(r'AH\d{5}', l))
print(f"AH##### 코드 포함 라인: {has_code}/{len(lines)}")

no_code_samples = [l for l in lines if not re.search(r'AH\d{5}', l)][:10]
for s in no_code_samples:
    print(s)

AH##### 코드 포함 라인: 629/629


In [15]:
ERROR_COLUMNS = [
    "line_no",
    "raw_line",

    "time_raw",
    "time",

    "module",
    "level",

    "pid",
    "tid",

    "client_ip",
    "client_port",

    "error_code",
    "message",

    "parse_status",
    "record_type",
    "parse_note",
]

In [16]:
STANDARD_ERROR_PATTERN = re.compile(
    r'^\[(?P<time_raw>[^\]]+)\]\s+'
    r'\[(?P<module>[^:\]]+):(?P<level>[^\]]+)\]\s+'
    r'\[pid\s+(?P<pid>\d+)'
    r'(?::tid\s+(?P<tid>\d+))?\]\s*'
    r'(?:\[client\s+(?P<client_raw>[^\]]+)\]\s*)?'
    r'(?P<message>.*)$'
)


ERROR_CODE_PATTERN = re.compile(
    r'\bAH(?P<error_code>\d{5})\b'
)

In [17]:
def parse_timestamp(value):
    """
    Apache Error Log timestamp를 pandas Timestamp로 변환한다.

    지원 형식:
    - Thu Jun 12 13:35:25.818494 2025
    - Thu Jun 12 13:35:25 2025
    """

    if value is None:
        return pd.NaT

    formats = [
        "%a %b %d %H:%M:%S.%f %Y",
        "%a %b %d %H:%M:%S %Y",
    ]

    for time_format in formats:
        try:
            parsed = datetime.strptime(
                value,
                time_format,
            )

            return pd.Timestamp(parsed)

        except ValueError:
            continue

    return pd.NaT

In [18]:
def parse_client(value):
    """
    client 필드에서 IP와 port를 분리한다.

    예:
    210.110.68.35:59629
    -> client_ip   = 210.110.68.35
    -> client_port = 59629

    IPv4와 IPv6 형식을 최대한 보존한다.
    """

    if value is None:
        return None, None

    value = value.strip()

    if value == "":
        return None, None

    # [IPv6]:port 형태
    if value.startswith("[") and "]" in value:
        bracket_end = value.find("]")

        ip_part = value[1:bracket_end]
        remaining = value[bracket_end + 1:]

        if (
            remaining.startswith(":")
            and remaining[1:].isdigit()
        ):
            port = int(remaining[1:])
        else:
            port = None

        return ip_part, port

    # Port가 없는 순수 IP
    try:
        ipaddress.ip_address(value)

        return value, None

    except ValueError:
        pass

    # IPv4:port 또는 IPv6:port 후보
    if ":" in value:
        ip_part, port_part = value.rsplit(
            ":",
            maxsplit=1,
        )

        if port_part.isdigit():
            try:
                ipaddress.ip_address(ip_part)

                return ip_part, int(port_part)

            except ValueError:
                pass

    # 완전히 분리하지 못하더라도 원본 값은 보존
    return value, None

In [19]:
def create_empty_record(
    line_no,
    raw_line,
):
    """
    모든 컬럼이 들어 있는 기본 레코드를 생성한다.

    파싱에 실패하더라도 DataFrame의 컬럼 구조가
    달라지지 않도록 한다.
    """

    record = {
        column: None
        for column in ERROR_COLUMNS
    }

    record["line_no"] = line_no
    record["raw_line"] = raw_line

    return record

In [20]:
def parse_error_line(
    line,
    line_no,
):
    """
    error log 한 줄을 구조화된 dict로 변환한다.
    """

    raw_line = line.rstrip("\r\n")

    record = create_empty_record(
        line_no=line_no,
        raw_line=raw_line,
    )

    # 빈 줄
    if raw_line == "":
        record.update({
            "parse_status": "FAIL",
            "record_type": "EMPTY_LINE",
            "parse_note": "EMPTY_LINE",
        })

        return record

    standard_match = (
        STANDARD_ERROR_PATTERN.fullmatch(
            raw_line
        )
    )

    error_code_match = (
        ERROR_CODE_PATTERN.search(
            raw_line
        )
    )

    error_code = (
        f"AH{error_code_match.group('error_code')}"
        if error_code_match
        else None
    )

    # 표준 Apache Error Log 형식
    if standard_match is not None:
        data = standard_match.groupdict()

        parsed_time = parse_timestamp(
            data["time_raw"]
        )

        client_ip, client_port = parse_client(
            data["client_raw"]
        )

        notes = []

        if pd.isna(parsed_time):
            notes.append(
                "INVALID_TIMESTAMP"
            )

        parse_status = (
            "PARTIAL"
            if notes
            else "SUCCESS"
        )

        record.update({
            "time_raw": data["time_raw"],
            "time": parsed_time,

            "module": data["module"],
            "level": data["level"],

            "pid": int(data["pid"]),

            "tid": (
                int(data["tid"])
                if data["tid"]
                else None
            ),

            "client_ip": client_ip,
            "client_port": client_port,

            "error_code": error_code,
            "message": data["message"],

            "parse_status": parse_status,
            "record_type": "STANDARD_HEADER",

            "parse_note": (
                ";".join(notes)
                if notes
                else None
            ),
        })

        return record

    # Timestamp 등의 표준 헤더는 없지만
    # Apache error code가 있는 메시지
    if error_code is not None:
        record.update({
            "error_code": error_code,
            "message": raw_line,

            "parse_status": "PARTIAL",
            "record_type": "MISSING_TIMESTAMP_HEADER",
            "parse_note": "MISSING_TIMESTAMP_HEADER",
        })

        return record

    # Error code도 없고 표준 헤더도 없는 행
    # continuation일 수 있으므로 바로 INVALID로 확정하지 않는다.
    record.update({
        "message": raw_line,

        "parse_status": "PARTIAL",
        "record_type": "UNSTRUCTURED_LINE",
        "parse_note": "UNSTRUCTURED_OR_CONTINUATION",
    })

    return record

In [21]:
def parse_error_file(file_path):
    """
    error log 전체를 읽어 DataFrame으로 반환한다.

    빈 행도 제거하지 않고 원본 행 번호를 그대로 보존한다.
    """

    file_path = Path(file_path)

    records = []

    with file_path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as file:

        for line_no, line in enumerate(
            file,
            start=1,
        ):
            record = parse_error_line(
                line=line,
                line_no=line_no,
            )

            missing_keys = (
                set(ERROR_COLUMNS)
                - set(record.keys())
            )

            unexpected_keys = (
                set(record.keys())
                - set(ERROR_COLUMNS)
            )

            assert not missing_keys, (
                f"{line_no}번째 레코드에 누락된 키가 있습니다: "
                f"{missing_keys}"
            )

            assert not unexpected_keys, (
                f"{line_no}번째 레코드에 예상하지 않은 키가 있습니다: "
                f"{unexpected_keys}"
            )

            records.append(record)

    frame = pd.DataFrame.from_records(
        records,
        columns=ERROR_COLUMNS,
    )

    integer_columns = [
        "line_no",
        "pid",
        "tid",
        "client_port",
    ]

    for column in integer_columns:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        ).astype("Int64")

    return frame

In [22]:
df = parse_error_file(
    error_file_path
)


print(f"전체 로그 수: {len(df):,}")

display(
    df.head(10)
)

전체 로그 수: 629


,line_no,raw_line,time_raw,time,module,level,pid,tid,client_ip,client_port,error_code,message,parse_status,record_type,parse_note
0,1,[Thu Jun 12 13:35:25.818494 2025] [suexec:notice] [pid 1947900:tid 140420159084864] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec),Thu Jun 12 13:35:25.818494 2025,2025-06-12 13:35:25.818494,suexec,notice,1947900,140420159084864,None,<NA>,AH01232,AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec),SUCCESS,STANDARD_HEADER,None
1,2,"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive...",None,NaT,None,None,<NA>,<NA>,None,<NA>,AH00558,"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive...",PARTIAL,MISSING_TIMESTAMP_HEADER,MISSING_TIMESTAMP_HEADER
2,3,[Thu Jun 12 13:35:25.840609 2025] [lbmethod_heartbeat:notice] [pid 1947900:tid 140420159084864] AH02282: No slotmem from mod_heartmonitor,Thu Jun 12 13:35:25.840609 2025,2025-06-12 13:35:25.840609,lbmethod_heartbeat,notice,1947900,140420159084864,None,<NA>,AH02282,AH02282: No slotmem from mod_heartmonitor,SUCCESS,STANDARD_HEADER,None
3,4,[Thu Jun 12 13:35:25.841319 2025] [http2:warn] [pid 1947900:tid 140420159084864] AH02951: mod_ssl does not seem to be enabled,Thu Jun 12 13:35:25.841319 2025,2025-06-12 13:35:25.841319,http2,warn,1947900,140420159084864,None,<NA>,AH02951,AH02951: mod_ssl does not seem to be enabled,SUCCESS,STANDARD_HEADER,None
4,5,[Thu Jun 12 13:35:25.843764 2025] [mpm_event:notice] [pid 1947900:tid 140420159084864] AH00489: Apache/2.4.37 (Rocky Linux) configured -- resuming...,Thu Jun 12 13:35:25.843764 2025,2025-06-12 13:35:25.843764,mpm_event,notice,1947900,140420159084864,None,<NA>,AH00489,AH00489: Apache/2.4.37 (Rocky Linux) configured -- resuming normal operations,SUCCESS,STANDARD_HEADER,None
5,6,[Thu Jun 12 13:35:25.843784 2025] [core:notice] [pid 1947900:tid 140420159084864] AH00094: Command line: '/usr/sbin/httpd -D FOREGROUND',Thu Jun 12 13:35:25.843784 2025,2025-06-12 13:35:25.843784,core,notice,1947900,140420159084864,None,<NA>,AH00094,AH00094: Command line: '/usr/sbin/httpd -D FOREGROUND',SUCCESS,STANDARD_HEADER,None
6,7,[Thu Jun 12 13:35:37.052218 2025] [autoindex:error] [pid 1947909:tid 140419562436352] [client 210.110.68.35:59629] AH01276: Cannot serve directory...,Thu Jun 12 13:35:37.052218 2025,2025-06-12 13:35:37.052218,autoindex,error,1947909,140419562436352,210.110.68.35,59629,AH01276,"AH01276: Cannot serve directory /home/dsuser/: No matching DirectoryIndex (index.html) found, and server-generated directory index forbidden by Op...",SUCCESS,STANDARD_HEADER,None
7,8,[Thu Jun 12 16:43:50.991473 2025] [autoindex:error] [pid 1947909:tid 140419308443392] [client 210.110.68.35:56126] AH01276: Cannot serve directory...,Thu Jun 12 16:43:50.991473 2025,2025-06-12 16:43:50.991473,autoindex,error,1947909,140419308443392,210.110.68.35,56126,AH01276,"AH01276: Cannot serve directory /home/dsuser/: No matching DirectoryIndex (index.html) found, and server-generated directory index forbidden by Op...",SUCCESS,STANDARD_HEADER,None
8,9,[Thu Jun 12 16:56:59.210399 2025] [autoindex:error] [pid 1947909:tid 140418427635456] [client 210.110.64.41:32198] AH01276: Cannot serve directory...,Thu Jun 12 16:56:59.210399 2025,2025-06-12 16:56:59.210399,autoindex,error,1947909,140418427635456,210.110.64.41,32198,AH01276,"AH01276: Cannot serve directory /home/dsuser/: No matching DirectoryIndex (index.html) found, and server-generated directory index forbidden by Op...",SUCCESS,STANDARD_HEADER,None
9,10,[Thu Jun 12 16:58:25.662484 2025] [autoindex:error] [pid 1948138:tid 140419182618368] [client 220.67.241.146:55672] AH01276: Cannot serve director...,Thu Jun 12 16:58:25.662484 2025,2025-06-12 16:58:25.662484,autoindex,error,1948138,140419182618368,220.67.241.146,55672,AH01276,"AH01276: Cannot serve directory /home/dsuser/: No matching

In [23]:
def make_distribution(
    frame,
    column,
):
    """
    컬럼별 건수와 비율을 계산한다.
    """

    result = (
        frame[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )

    if len(frame) == 0:
        result["rate_pct"] = 0.0
    else:
        result["rate_pct"] = (
            result["count"]
            / len(frame)
            * 100
        ).round(2)

    return result


parse_summary = make_distribution(
    df,
    "parse_status",
)

record_type_summary = make_distribution(
    df,
    "record_type",
)


print("[Parse Status]")

display(
    parse_summary
)


print("[Record Type]")

display(
    record_type_summary
)

[Parse Status]


,parse_status,count,rate_pct
0,SUCCESS,611,97.14
1,PARTIAL,18,2.86


[Record Type]


,record_type,count,rate_pct
0,STANDARD_HEADER,611,97.14
1,MISSING_TIMESTAMP_HEADER,18,2.86


In [24]:
sample_columns = [
    "line_no",
    "time_raw",
    "module",
    "level",
    "error_code",
    "parse_status",
    "record_type",
    "parse_note",
    "raw_line",
]


partial_rows = df.loc[
    df["parse_status"].eq("PARTIAL"),
    sample_columns,
]


failure_rows = df.loc[
    df["parse_status"].eq("FAIL"),
    sample_columns,
]


print("[PARTIAL 예시]")
print(f"전체 건수: {len(partial_rows):,}")

display(
    partial_rows.head(20)
)


print("[FAIL 예시]")
print(f"전체 건수: {len(failure_rows):,}")

display(
    failure_rows.head(20)
)

[PARTIAL 예시]
전체 건수: 18


,line_no,time_raw,module,level,error_code,parse_status,record_type,parse_note,raw_line
1,2,None,None,None,AH00558,PARTIAL,MISSING_TIMESTAMP_HEADER,MISSING_TIMESTAMP_HEADER,"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive..."
14,15,None,None,None,AH00558,PARTIAL,MISSING_TIMESTAMP_HEADER,MISSING_TIMESTAMP_HEADER,"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive..."
21,22,None,None,None,AH00558,PARTIAL,MISSING_TIMESTAMP_HEADER,MISSING_TIMESTAMP_HEADER,"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive..."
28,29,None,None,None,AH00558,PARTIAL,MISSING_TIMESTAMP_HEADER,MISSING_TIMESTAMP_HEADER,"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive..."
35,36,None,None,None,AH00558,PARTIAL,MISSING_TIMESTAMP_HEADER,MISSING_TIMESTAMP_HEADER,"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive..."
42,43,None,None,None,AH00558,PARTIAL,MISSING_TIMESTAMP_HEADER,MISSING_TIMESTAMP_HEADER,"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive..."
53,54,None,None,None,AH00558,PARTIAL,MISSING_TIMESTAMP_HEADER,MISSING_TIMESTAMP_HEADER,"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive..."
66,67,None,None,None,AH00558,PARTIAL,MISSING_TIMESTAMP_HEADER,MISSING_TIMESTAMP_HEADER,"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive..."
87,88,None,None,None,AH00558,PARTIAL,MISSING_TIMESTAMP_HEADER,MISSING_TIMESTAMP_HEADER,"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive..."
104,105,None,None,None,AH00558,PARTIAL,MISSING_TIMESTAMP_HEADER,MISSING_TIMESTAMP_HEADER,"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive..."


[FAIL 예시]
전체 건수: 0


,line_no,time_raw,module,level,error_code,parse_status,record_type,parse_note,raw_line


In [25]:
level_summary = make_distribution(
    df.loc[df["level"].notna()],
    "level",
)

module_summary = make_distribution(
    df.loc[df["module"].notna()],
    "module",
)


print("[Level Distribution]")

display(
    level_summary
)


print("[Module Distribution]")

display(
    module_summary
)

[Level Distribution]


,level,count,rate_pct
0,warn,405,66.28
1,error,117,19.15
2,notice,89,14.57


[Module Distribution]


,module,count,rate_pct
0,proxy,442,72.34
1,proxy_http,48,7.86
2,mpm_event,35,5.73
3,core,26,4.26
4,suexec,18,2.95
5,lbmethod_heartbeat,18,2.95
6,http2,18,2.95
7,autoindex,6,0.98


In [26]:
timestamp_missing_records = []


timestamp_missing_indexes = df.index[
    df["record_type"].eq(
        "MISSING_TIMESTAMP_HEADER"
    )
]


for index in timestamp_missing_indexes:
    previous_line = None
    next_line = None

    if index > 0:
        previous_line = df.iloc[
            index - 1
        ]["raw_line"]

    if index + 1 < len(df):
        next_line = df.iloc[
            index + 1
        ]["raw_line"]

    timestamp_missing_records.append({
        "line_no": int(
            df.at[index, "line_no"]
        ),

        "previous_line": previous_line,

        "raw_line": df.at[
            index,
            "raw_line",
        ],

        "next_line": next_line,

        "error_code": df.at[
            index,
            "error_code",
        ],

        "parse_note": df.at[
            index,
            "parse_note",
        ],
    })


timestamp_missing_df = pd.DataFrame(
    timestamp_missing_records,
    columns=[
        "line_no",
        "previous_line",
        "raw_line",
        "next_line",
        "error_code",
        "parse_note",
    ],
)


print(
    "Timestamp 없는 행:",
    len(timestamp_missing_df),
)


display(
    timestamp_missing_df.head(10)
)

Timestamp 없는 행: 18


,line_no,previous_line,raw_line,next_line,error_code,parse_note
0,2,[Thu Jun 12 13:35:25.818494 2025] [suexec:notice] [pid 1947900:tid 140420159084864] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec),"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive...",[Thu Jun 12 13:35:25.840609 2025] [lbmethod_heartbeat:notice] [pid 1947900:tid 140420159084864] AH02282: No slotmem from mod_heartmonitor,AH00558,MISSING_TIMESTAMP_HEADER
1,15,[Thu Jun 12 17:42:00.043754 2025] [suexec:notice] [pid 1995107:tid 140658241005888] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec),"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive...",[Thu Jun 12 17:42:00.056234 2025] [lbmethod_heartbeat:notice] [pid 1995107:tid 140658241005888] AH02282: No slotmem from mod_heartmonitor,AH00558,MISSING_TIMESTAMP_HEADER
2,22,[Thu Jun 12 17:54:44.103613 2025] [suexec:notice] [pid 1995750:tid 140694063937856] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec),"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive...",[Thu Jun 12 17:54:44.116054 2025] [lbmethod_heartbeat:notice] [pid 1995750:tid 140694063937856] AH02282: No slotmem from mod_heartmonitor,AH00558,MISSING_TIMESTAMP_HEADER
3,29,[Thu Jun 12 17:56:07.326271 2025] [suexec:notice] [pid 1996011:tid 140669294074176] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec),"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive...",[Thu Jun 12 17:56:07.339098 2025] [lbmethod_heartbeat:notice] [pid 1996011:tid 140669294074176] AH02282: No slotmem from mod_heartmonitor,AH00558,MISSING_TIMESTAMP_HEADER
4,36,[Thu Jun 12 17:57:37.322539 2025] [suexec:notice] [pid 1996265:tid 139695085410624] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec),"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive...",[Thu Jun 12 17:57:37.335006 2025] [lbmethod_heartbeat:notice] [pid 1996265:tid 139695085410624] AH02282: No slotmem from mod_heartmonitor,AH00558,MISSING_TIMESTAMP_HEADER
5,43,[Thu Jun 12 17:58:37.904591 2025] [suexec:notice] [pid 1996504:tid 140592679917888] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec),"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive...",[Thu Jun 12 17:58:37.917365 2025] [lbmethod_heartbeat:notice] [pid 1996504:tid 140592679917888] AH02282: No slotmem from mod_heartmonitor,AH00558,MISSING_TIMESTAMP_HEADER
6,54,[Mon Jun 16 14:47:19.351148 2025] [suexec:notice] [pid 2065903:tid 140235550566720] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec),"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive...",[Mon Jun 16 14:47:19.363946 2025] [lbmethod_heartbeat:notice] [pid 2065903:tid 140235550566720] AH02282: No slotmem from mod_heartmonitor,AH00558,MISSING_TIMESTAMP_HEADER
7,67,[Mon Jun 16 15:33:18.474815 2025] [suexec:notice] [pid 971:tid 140566996662592] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec),"AH00558: httpd: Could not reliably determine the server's fully qualified domain name, using localhost.localdomain. Set the 'ServerName' directive...",[Mon Jun 16 15:33:18.501914 2025] [lbmethod_heartbeat:notice] [pid 971:tid 140566996662592] AH02282: No slotmem from mod_heartmonitor,AH00558,MISSING_TIMESTAMP_HEADER
8,88,[Mon Jun 16 15:47:24.759522 2025] [suexec:notice] [pid 8619:tid 139944384870720] AH01232: suEXEC mechanism enabled (wrapper: /usr/sbin/suexec),"AH00

In [27]:
# 전체 629행 저장
df.to_csv(
    parsed_csv_path,
    index=False,
    encoding="utf-8-sig",
)


# 진짜 FAIL 행만 별도 저장
failure_columns = [
    "line_no",
    "raw_line",
    "parse_status",
    "record_type",
    "parse_note",
]


failures = df.loc[
    df["parse_status"].eq("FAIL"),
    failure_columns,
].copy()


failures.to_csv(
    failure_csv_path,
    index=False,
    encoding="utf-8-sig",
)


# Timestamp가 없는 행과 앞뒤 문맥 저장
timestamp_missing_df.to_csv(
    timestamp_missing_csv_path,
    index=False,
    encoding="utf-8-sig",
)


print(
    f"저장 완료: {parsed_csv_path} "
    f"({len(df):,}행)"
)

print(
    f"저장 완료: {failure_csv_path} "
    f"({len(failures):,}행)"
)

print(
    f"저장 완료: {timestamp_missing_csv_path} "
    f"({len(timestamp_missing_df):,}행)"
)

저장 완료: C:\Users\seoyeon\log-dlq-pipeline\data\interim\error_parsed.csv (629행)
저장 완료: C:\Users\seoyeon\log-dlq-pipeline\data\output\error_parse_failures.csv (0행)
저장 완료: C:\Users\seoyeon\log-dlq-pipeline\data\output\error_timestamp_missing.csv (18행)


In [28]:
with error_file_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
) as file:
    raw_line_count = sum(
        1
        for _ in file
    )


# 원본 행 수 보존
assert len(df) == raw_line_count, (
    "원본 행 수와 DataFrame 행 수가 다릅니다."
)


# 컬럼과 순서 확인
assert df.columns.tolist() == ERROR_COLUMNS, (
    "DataFrame 컬럼 또는 컬럼 순서가 ERROR_COLUMNS와 다릅니다."
)


# line_no는 1부터 시작하고 원본 순서와 일치
assert df["line_no"].is_unique, (
    "line_no가 중복되었습니다."
)

assert df["line_no"].tolist() == list(
    range(1, len(df) + 1)
), (
    "line_no가 원본 행 순서와 일치하지 않습니다."
)


# 원본 문자열 보존
assert df["raw_line"].notna().all(), (
    "raw_line에 결측치가 있습니다."
)


# Parser 상태 검증
allowed_parse_status = {
    "SUCCESS",
    "PARTIAL",
    "FAIL",
}

assert df["parse_status"].isin(
    allowed_parse_status
).all(), (
    "허용되지 않은 parse_status가 있습니다."
)


# SUCCESS는 timestamp가 존재해야 함
success_timestamp_valid = df.loc[
    df["parse_status"].eq("SUCCESS"),
    "time",
].notna().all()

assert success_timestamp_valid, (
    "SUCCESS 행 중 timestamp 변환에 실패한 행이 있습니다."
)


# Timestamp 헤더가 없는 행은 PARTIAL이어야 함
missing_header_status_valid = df.loc[
    df["record_type"].eq(
        "MISSING_TIMESTAMP_HEADER"
    ),
    "parse_status",
].eq("PARTIAL").all()

assert missing_header_status_valid, (
    "Timestamp 없는 행이 PARTIAL로 분류되지 않았습니다."
)


# FAIL CSV 건수 일치
assert len(failures) == int(
    df["parse_status"].eq("FAIL").sum()
), (
    "FAIL 건수와 실패 CSV 건수가 다릅니다."
)


# Timestamp 누락 CSV 건수 일치
assert len(timestamp_missing_df) == int(
    df["record_type"]
    .eq("MISSING_TIMESTAMP_HEADER")
    .sum()
), (
    "Timestamp 누락 건수가 일치하지 않습니다."
)


print("기본 검증 통과")

print(
    f"- 원본/DF 행 수: {raw_line_count:,}"
)

print(
    f"- line_no 고유성: {df['line_no'].is_unique}"
)

print(
    f"- 최종 컬럼 수: {len(df.columns)}"
)

기본 검증 통과
- 원본/DF 행 수: 629
- line_no 고유성: True
- 최종 컬럼 수: 15


In [29]:
df.info()

df.columns.tolist()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 629 entries, 0 to 628
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   line_no       629 non-null    Int64         
 1   raw_line      629 non-null    object        
 2   time_raw      611 non-null    object        
 3   time          611 non-null    datetime64[ns]
 4   module        611 non-null    object        
 5   level         611 non-null    object        
 6   pid           611 non-null    Int64         
 7   tid           611 non-null    Int64         
 8   client_ip     449 non-null    object        
 9   client_port   449 non-null    Int64         
 10  error_code    629 non-null    object        
 11  message       629 non-null    object        
 12  parse_status  629 non-null    object        
 13  record_type   629 non-null    object        
 14  parse_note    18 non-null     object        
dtypes: Int64(4), datetime64[ns](1), object(1

['line_no',
 'raw_line',
 'time_raw',
 'time',
 'module',
 'level',
 'pid',
 'tid',
 'client_ip',
 'client_port',
 'error_code',
 'message',
 'parse_status',
 'record_type',
 'parse_note']